In [ ]:
import math
import os

from pyspark.sql import functions as F
from pyspark.sql import types as T
from sparkgeo import functions as S
from sparkgeo import processors as P

In [ ]:
pipelines = (
    spark.read.parquet(os.path.expanduser("~/pipelines.prq"))
    # rename
    .withColumnRenamed("Shape@WKB", "geom")
)

In [ ]:
pipelines.count()

In [ ]:
pipelines.printSchema()

In [ ]:
((ymin, xmin), (ymax, xmax)) = pipelines.st_bounds()

In [ ]:
cell = math.ceil(min(xmax - xmin, ymax - ymin) / 30.0)
cell

In [ ]:
lhs = pipelines.select(F.col("OBJECTID").alias("lid"), "geom")
rhs = pipelines.select(F.col("OBJECTID").alias("rid"), "geom")

In [ ]:
df = (
    lhs.join_qr(rhs, cell=cell)
    .filter("lid != rid")
    .filter(S.st_intersects("lgeom", "rgeom"))
)

In [ ]:
df.count()

In [ ]:
df.printSchema()

In [ ]:
# lid_count = df.select("lid").groupBy("lid").count().orderBy(F.col("count").desc())

In [ ]:
lid = df.select(F.col("lid").alias("ROBJECTID")).distinct()

In [ ]:
(
    pipelines.join(lid, pipelines.OBJECTID == lid.ROBJECTID, how="left")
    .filter(F.col("ROBJECTID").isNull())
    .drop("ROBJECTID")
    .select("OBJECTID")
    .show()
)